# KV Cache Lab：先理解源码 TODO，再研究 Radix 树与 LFU

目标是先把 HiCache 与 MLA 两个源码 TODO 的当前缺口、调用链和最小验证负载讲清，再用 Radix 树与资源账本实验建立基础。完整 HiCache/MLA、LFU 实现是大改理论题，转去 SGLang 主仓库后再按真实需求选题。实现由你完成；这里给可观察要求、负载、问题和阅读入口，不给补丁。

**样例约定**：本 notebook 的“做完后应看到”描述验收后的可观察行为，不是本次服务器实测结果。省略初始化与时间戳；`<...>` 是要用实际值填写的位置。性能数字不设固定目标；带示意数值的性能样例仅用于说明格式。实现题须运行验收，理论题还须交书面记录。


## 任务标签与转场规则

- **【必做】**：在 mini 完成调用链、真实轨迹或对拍；这些组成转去 SGLang 主仓库的门槛。
- **【小改选做】**：改动范围局部，适合加深理解；做一两项即可，不必全部完成。
- **【大改理论】**：保留原题的源码入口与验收设想，只交边界分析、反例和可复现测试计划；完整实现应在确认上游需求后进入 SGLang 主仓库。

完成本 notebook 的【必做】后，继续做 [lab 首页](../README.md) 所列其余章节的【必做】；全部完成便转去 SGLang 主仓库选题，无须等选做和大改题的完整实现。


## 【必做】先做源码 TODO 基础：HiCache 与 MLA

mini-sglang 在 [`MatchResult`](../../python/minisgl/kvcache/base.py) 明确留下 `TODO: support HiCache`，在 [`create_kvcache_pool`](../../python/minisgl/kvcache/__init__.py) 留下 `TODO: support other variants (e.g. MLA)`。这两个是本 lab **首先要分析的源码缺口**，不是末尾的补充阅读。H0/M0 是必做基础；H1/M1/M2 保留完整实现的验收标准，但在 mini 只交理论分析与测试计划。后面的原有 Phase 0、Phase 3 用来补足 radix/资源账本知识；其余 Phase 按标签选做。

| 优先级 | 源码 TODO | 要交付的行为证据 | 完成判定 |
|---|---|---|---|
| H | `MatchResult` / `PrefillAdder` 的 HiCache、host match | H0 必做：定位 GPU 命中与 host 缺口 | H1 大改理论：描述逐出、host 恢复和正确性验收 |
| M | KV pool 工厂的 MLA 变体 | M0 必做：定位 MHA 假设与 MLA 缺口 | M1/M2 大改理论：描述压缩布局、后端消费和数值对拍 |

H 与 M 的理论方案分别写；不要求在 mini 实现，更不要求组合成“MLA + HiCache”。原有 TP=1、`page_size=1`、关闭 CUDA Graph 的边界保留为将来验证方案的起点。


### 【必做】H0：HiCache 的当前缺口与失败基线

从 [`MatchResult`](../../python/minisgl/kvcache/base.py) → [`CacheManager.match_req`](../../python/minisgl/scheduler/cache.py) → [`PrefillAdder._try_allocate_one`](../../python/minisgl/scheduler/prefill.py) 画调用链。记录：当前只返回 `cuda_handle`；prefill 准入只读取 GPU 命中；[`RadixPrefixCache.evict`](../../python/minisgl/kvcache/radix_cache.py) 归还 page index 后不保留 host 层 KV。再沿 [`MHAKVCache`](../../python/minisgl/kvcache/mha_pool.py) 确定需要搬运的层、K/V 张量与 page 布局。

**必做证据**：用原有 radix/cache probe 记录一次 GPU 前缀命中与一次 evict，指出当前 `MatchResult` 和 prefill 准入为什么只能利用 GPU 命中。设计一组“暖前缀 → 施加 page 压力 → 重发前缀”的请求级负载，写出应采集的命中长度、prefill token、输出和 page 账本；服务器条件允许时可运行这组现状基线。无需先做 LFU。


### 【大改理论】H1：理论分析 GPU ↔ host 前缀复用

**本路线只交设计图、资源不变量和失败/通过测试计划。**下面的实现要求保留作将来在上游选中对应任务时的验收参考。


**最小范围**：提供有容量上限的 host KV 层；GPU 上可逐出的完整 page 在被重用前可转存到 host，并保留 token 前缀到 host 数据的可匹配关系。`MatchResult` 能区分 GPU 已命中的前缀和仅在 host 命中的延长部分；调度准入必须把 host 命中、GPU 可用 page、搬运完成时机和本轮 token budget 一起计算。host 数据恢复到新的 GPU page 后，更新请求 page table，attention 才能读；`cached_len` 只能计入真正可用的 KV。首个可运行版本可同步搬运，之后再测异步重叠。

**不可破坏的不变量**：

- 锁定的 GPU page 在使用期间不能被逐出；host 条目在恢复完成前不能被淘汰或复用。GPU 与 host 的 page 所有权、锁和释放次数都可核对。
- prefix 长度按 page size 对齐；只有完整、可恢复的页才算 host hit。host 容量满时可以淘汰 host 条目，之后请求应退回正常 prefill。
- 关闭 HiCache 时，现有纯 GPU radix 命中、evict、abort 与输出保持原行为。不要把“host 有 token id”误当成“host 有每层 KV”。

**验收**：重复 H0 的三请求轨迹，逐出后须出现 `host_hit_tokens > 0`、真实 H2D KV 搬运和比冷请求更少的模型 prefill token；逐 token 贪心输出与未逐出基线一致。再做 host 容量满、被锁前缀不可逐出、请求中止三组边界，逐次核对 GPU 空闲页 + 缓存页、host 占用量和释放次数。只增加字段或只缓存 token ids 不通过。


### 【必做】M0：MLA pool 的当前缺口与失败基线

先从 [`create_kvcache_pool`](../../python/minisgl/kvcache/__init__.py) 读出当前无条件构造 `MHAKVCache`，再检查 [`BaseKVCachePool`](../../python/minisgl/kvcache/base.py) 的 `k_cache/v_cache/store_kv`、[`ModelConfig`](../../python/minisgl/models/config.py) 的模型元数据、[`AttentionLayer`](../../python/minisgl/layers/attention.py) 的 Q/K/V 拆分，以及 [`Engine._determine_num_pages`](../../python/minisgl/engine/engine.py) 的每页显存估算。写下哪些契约假设了“分开的 K/V heads × head_dim”。

**必做证据**：画出 `ModelConfig → create_kvcache_pool → AttentionLayer → backend` 的形状与容量假设表，指出当前为何固定走 MHA pool；列出 MLA 需要的新数据表示及至少一个可推翻“只换 pool 类就够了”的反例。选择真实 MLA checkpoint、跑参考 greedy 输出和记录启动失败可作为后续上游任务的准备，不是转场门槛。


### 【大改理论】M1：理论分析 MLA KV pool 的选择与布局

**本路线只交 pool 契约、布局/字节数计算和合成张量测试计划。**完整实现和中间验收标准保留作未来上游任务的参考。


让模型配置驱动 KV pool 类型选择；MLA 路径保存目标模型真实需要的压缩 KV 表示与位置相关分量，并有正确的按层、按物理位置写入/读取接口。不要把两个普通 MHA 张量改名为 MLA，也不要让已有 MHA 模型误选新 pool。若现有 `BaseKVCachePool` 接口不能表达目标布局，应先写出失败的张量形状对拍，再作最小接口调整。同步修正 page 容量估算；使用的单位要与 [`CacheManager`](../../python/minisgl/scheduler/cache.py) 的物理 page 和全局 page table 一致。

**中间验收**：用小的确定性张量覆盖两层、非连续物理 page index、重复写入同一位置和 page 边界，逐位置读回的数据与写入值一致；分配/释放 page 后没有串层或越界。记录实际每 token KV 字节数，并与同配置的未压缩 K/V 表示比较。MHA pool 原有 store/read 行为和缓存测试必须仍通过。这个检查点只证明 pool，**不等于模型已能使用 MLA**。


### 【大改理论】M2：理论分析端到端消费 MLA 缓存

**本路线只交模型→attention→物理 page 的调用图及对拍计划。**真实模型接入属于大改，在 SGLang 主仓库有明确需求后再做。


接通所选模型的投影/位置编码、attention backend 和 KV pool，让 prefill 与 decode 真正读写 M1 的表示；完成一次有共享前缀的请求，验证第二次请求从该 pool 的物理 page 复用已写入 KV。验收时须记录 `pool 类型 → 每层写入形状 → page table 位置 → backend 实际读取 → logits` 的路径，不能只用类名或内存占用证明。

**完成条件**：同一 input ids 下，短样例至少 12 个 greedy token 与所选参考模型一致；冷请求、共享前缀热请求和接近 page 边界的请求均正确；真实分配的 KV 字节数与 M1 的计算吻合；MHA 模型回归通过。若 backend 仍把 MLA 数据展开成普通 MHA KV，应明确记录实际额外显存与计算开销，不能声称已经实现压缩缓存收益。CUDA Graph、TP>1 和 MLA+HiCache 组合另列未验证范围。


### 【必做】TODO 基础的交付顺序

1. 完成 H0/M0 的源码调用链、现状证据与最小反例。
2. 运行下方 Phase 0 与 Phase 3，交 radix 树图和 page 分配/回收账本。
3. H1/M1/M2 只提交接口与资源不变量、失败/通过负载、未验证边界，然后按 lab 首页转去上游选题。

下方原有实验完整保留，用于补齐树结构、锁和淘汰策略。小改题任选；LFU 的完整策略接入只作理论分析。


## 原有 Radix/LFU 实验如何支撑 TODO 主线

下面的原有 Phase 0–4 保留为基础与进阶练习。mini 已有前缀匹配、节点拆分、锁定、page 分配和 LRU 淘汰，而 `check_integrity` 尚无行为；这些练习能帮助检查 H 的 page 所有权与锁，也能为 M 的 page 布局回归提供基线。LFU 的完整策略接入作为大改理论题，优先级低于源码 TODO 基础。

| 阶段 | 你要做的事 | 验收 |
|---|---|---|
| 0. 观察树 | 预测共享前缀、分支、锁定、页对齐和淘汰的状态 | 现有代码应通过 |
| 1. 树内小改动 | 让 `check_integrity` 识别正常树和两类损坏 | 健康树通过，计数与拓扑损坏被拒绝 |
| 2. 对照魔改 | 暂时让默认淘汰偏向最近访问的分支，再恢复 | 同一负载选出的分支改变 |
| 3. 资源账本 | 追踪 `CacheManager` 的 page 分配与回收 | 现有行为测试通过 |
| 4. 主仓库功能 | 默认仍为 LRU，允许显式选择 LFU | 两种策略选择不同分支；锁定始终有效 |

请保留现有缓存测试。Phase 2 的临时改动在进入 Phase 4 前恢复；Phase 4 的实现应可被离线 `LLM` 配置使用。这些 Phase 的编号沿用原 lab，不代表应先于上面的 H/M TODO。


## 先分清三层

| mini 文件 | 读它要回答的问题 |
|---|---|
| `python/minisgl/kvcache/mha_pool.py` | 物理 K/V 张量在哪里？ |
| `python/minisgl/scheduler/cache.py`、`table.py` | 空闲 page、请求的 page table 和前缀树怎样交接？ |
| `python/minisgl/kvcache/radix_cache.py` | token 前缀与可复用 page index 怎样组织、保护和淘汰？ |
| `python/minisgl/scheduler/prefill.py`、`scheduler.py` | 请求何时查询/插入缓存、何时释放资源？ |

建议先读 `radix_cache.py` 的公开方法，再顺着 `CacheManager` 的调用链回到 scheduler。`python/minisgl/kernel/radix.py` 的比较实现、attention backend、CUDA graph、模型代码和主仓库 HiCache 在 H 任务中按需对照；本节只观察 mini 的现有 GPU radix 行为。


## 环境

前面的缓存结构与策略负载只需 CPU，但要在**已安装 mini-sglang 及其依赖**的环境、从仓库根目录运行；首次运行可能编译项目的 CPU 前缀比较扩展。完整 `LLM` 接入检查需 Linux + NVIDIA CUDA 服务器。`observe.py` 只驱动真实缓存对象和断言，不替你改核心代码。


## 【必做】Phase 0：先画出真正的 radix 树

画出 `insert_prefix` → `match_prefix` → `lock_handle` → `evict` → `unlock`。分别标出 token key、page indices、压缩边、树节点、handle 与 `evictable_size/protected_size` 由谁持有。

运行前预测：

1. 插入 `[11,12,13,14]` 与 `[11,12,15,16]`，两条分支的前两个 page indices 相同。树保存几个 token/page index？为什么不是八个？
2. 查询 `[11,12,99]` 后树是否可能改变？锁住返回的 handle，再淘汰四个 token，哪些 page indices 可以回收？解锁后又能回收什么？
3. `page_size=4` 时插入六个 token，能缓存的长度是多少？两个只在第四个 token 不同的输入能共享多少页？

请把每次操作后的树画成“边上的 token 片段 → page index 片段”，并记录每个节点的锁定状态；仅画命中长度不算完成。

### 本题文件导航

- **接口到树**：[python/minisgl/kvcache/base.py](../../python/minisgl/kvcache/base.py) → [python/minisgl/kvcache/radix_cache.py](../../python/minisgl/kvcache/radix_cache.py)。逐步追插入、匹配、锁定、解锁与淘汰；画出节点与 handle 的关系。
- **负载与验收**：[lab/kv_cache/observe.py](../kv_cache/observe.py)。先读本题对应的负载和断言，明确观测对象。
- **暂时跳过**：GPU KV 张量、attention kernel 和模型加载。

### 做完后应看到

三个 CPU probe 的关键输出应为：

```text
# radix-tree；两条分支的 evicted 顺序可能不同，验收比较集合
[branch] shared_len= 2 size= SizeInfo(evictable_size=6, protected_size=0)
[branch] evicted= [102, 103, 200, 201] size= SizeInfo(evictable_size=0, protected_size=2)
[check] PASS

# split-lock
[match] cached_len= 2
[before lock] size= SizeInfo(evictable_size=4, protected_size=0)
[locked] size= SizeInfo(evictable_size=2, protected_size=2)
[evicted] indices= [102, 103]
[check] PASS

# page
[page] hit= 4 miss= 0
[page] size= SizeInfo(evictable_size=4, protected_size=0)
[check] PASS
```

**完成效果**：日志里共享部分只算一次；锁住部分留下，非共享部分可回收；页对齐后只有完整页能命中。另交三份操作前后的树图，标明 edge key/value、锁定与 size 账本。

**失败时观察**：命中长度、回收 indices 或 size 与样例不符；每个 probe 自己打印一次 PASS，只看到最后一个 PASS 不能替代前面命令的结果。


In [ ]:
%%bash
export MINISGL_OBSERVE=0
python lab/kv_cache/observe.py --case radix-tree
python lab/kv_cache/observe.py --case split-lock
python lab/kv_cache/observe.py --case page


**记录**：把三次运行的命中长度、size 计数与回收的 page indices 记下来。解释分裂后共享前缀为何只计一次，以及锁住共享节点时两个尾部分支为何仍可淘汰。再追问：`match_prefix` 的接口文字与实际树结构变化完全一致吗？使用返回的 page indices 前为什么需要锁定 handle？


## 【小改选做】Phase 1：给 radix 树补一个结构完整性实验

`RadixPrefixCache.check_integrity()` 是 mini 已有的接口。请让它在正常的插入、节点拆分、锁定、解锁与淘汰后通过；若树的总量账本被人为改坏，或父子拓扑被人为断开，必须抛出错误。只检查 radix 树自身可以判断的事实；不要在这里重写 `CacheManager` 的空闲 page 检查，也不要改变插入/匹配/淘汰语义。

先运行下面的负载：它先构造共享前缀的分支，再锁住共享前缀，分别注入两种损坏，最后恢复并继续淘汰。未实现时，它应在第一种损坏上失败。你的实现让整个负载通过后，再重跑 Phase 0。

**先答后改**：压缩边的 key/value 长度有何关系？根节点和普通节点的计数含义是否相同？如何让一个分支的锁定不被误算为整个子树的保护？如果只比较 `size_info.total_size`，能否发现父子关系断裂？

阅读入口：mini 的 `python/minisgl/kvcache/base.py` 接口契约、`python/minisgl/kvcache/radix_cache.py` 的节点生命周期与公开方法、`python/minisgl/scheduler/cache.py` 对完整性检查的调用。主仓库只对照 [RadixCache](https://github.com/sgl-project/sglang/blob/main/python/sglang/srt/mem_cache/radix_cache.py) 的职责边界。暂时不用读 attention kernel、HiCache 和分布式代码。

### 本题文件导航

- **契约与拓扑**：[python/minisgl/kvcache/base.py](../../python/minisgl/kvcache/base.py) → [python/minisgl/kvcache/radix_cache.py](../../python/minisgl/kvcache/radix_cache.py)。列出树自身应维持的结构与计数关系，覆盖节点分裂和锁的生命周期。
- **调用边界**：[python/minisgl/scheduler/cache.py](../../python/minisgl/scheduler/cache.py)。确认树检查与资源管理器检查各负责什么。
- **负载与验收**：[lab/kv_cache/observe.py](../kv_cache/observe.py)。先读本题对应的负载和断言，明确观测对象。
- **暂时跳过**：重写空闲 page 分配、attention kernel 和 HiCache。

### 做完后应看到

未实现时，应能看到可观察的失败，例如：

```text
AssertionError: integrity check missed a corrupted size counter
```

完成后，同一 `integrity` 负载只在末尾打印：

```text
[check] PASS
```

**通过标准**：正常结构通过，两次人为损坏分别被你的检查拒绝，恢复之后继续通过；重跑 Phase 0 的三个 probe 也通过。probe 会捕获预期的损坏异常，所以成功时不会打印它们的 traceback。

**失败时观察**：若总量检查通过但父子拓扑没有被拒绝，会报 `integrity check missed a broken parent link`；仅给正常情况打印 PASS 不足以完成本题。


In [ ]:
%%bash
export MINISGL_OBSERVE=0
python lab/kv_cache/observe.py --case integrity
python lab/kv_cache/observe.py --case radix-tree


## 【小改选做】Phase 2：一次可恢复的极小魔改

下面的负载依次插入 A、B 两条独立分支，随后再访问 A，最后只淘汰一条分支。先运行默认行为，再暂时把默认淘汰规则改成**优先淘汰最近访问的可淘汰分支**；其余缓存语义、锁与 page 回收行为保持不变。让第二个命令通过后，恢复默认行为，并重新运行第一个命令。

动手前回答：插入顺序与最后访问顺序在这个负载中是否相同？如果相同，负载能否区分 LRU、MRU 与按插入顺序淘汰？淘汰内部节点与叶子节点的条件相同吗？

阅读入口：mini 的 radix 树节点与淘汰流程。不要为这个临时改动新增通用策略框架。

### 本题文件导航

- **先读**：[python/minisgl/kvcache/radix_cache.py](../../python/minisgl/kvcache/radix_cache.py)。追访问记录、候选遍历、锁定约束与淘汰的生命周期，解释负载怎样区分两种规则。
- **再对照**：[python/minisgl/kvcache/base.py](../../python/minisgl/kvcache/base.py)。确认临时对照仍满足返回回收 indices 的接口契约。
- **负载与验收**：[lab/kv_cache/observe.py](../kv_cache/observe.py)。先读本题对应的负载和断言，明确观测对象。
- **暂时跳过**：新策略框架、GPU attention 和模型前向。

### 做完后应看到

两份输出应只在淘汰的分支上体现策略差异：

```text
# 默认 LRU：复用 A 后淘汰 B
[eviction] recently_used=A, indices= [200, 201]
[eviction] size= SizeInfo(evictable_size=2, protected_size=0)
[check] PASS

# 临时 MRU：复用 A 后淘汰 A
[eviction] recently_used=A, indices= [100, 101]
[eviction] size= SizeInfo(evictable_size=2, protected_size=0)
[check] PASS
```

**通过标准**：各自在对应代码版本通过；恢复 LRU 后第一个命令再次通过。幸存分支仍命中长度 2，回收的是物理 indices。

**失败时观察**：indices 相同就没有展示所要求的差异；淘汰后无法匹配幸存分支，说明除了策略还破坏了缓存行为。


In [ ]:
%%bash
export MINISGL_OBSERVE=0
# 改动前以及恢复后
python lab/kv_cache/observe.py --case eviction --expect lru


In [ ]:
%%bash
export MINISGL_OBSERVE=0
# 临时魔改期间
python lab/kv_cache/observe.py --case eviction --expect mru


## 【必做】Phase 3：理解 CacheManager 的资源账本

在 mini 中，前缀树决定哪些缓存分支可回收；`CacheManager` 持有空闲 page 并按需请求淘汰。请从 `CacheManager._allocate` 追到 radix `evict`，再从请求完成路径追到 `cache_req` 和 `lazy_free_region`。

问题：树里的 `evictable_size` 与 `CacheManager.available_size` 为什么不是同一个数字？一次淘汰若回收了多于请求数量的 token，空闲 page 账本该怎样变化？`page_size > 1` 时哪些值必须保持对齐？先运行现有行为测试，再解释其中一次 allocate→evict→allocate 的账本。

### 本题文件导航

- **分配到回收**：[python/minisgl/scheduler/cache.py](../../python/minisgl/scheduler/cache.py) → [python/minisgl/kvcache/radix_cache.py](../../python/minisgl/kvcache/radix_cache.py)。画出缺页时的分配/淘汰路线，分别记录树和空闲 page 的账本。
- **请求生命周期**：[python/minisgl/scheduler/scheduler.py](../../python/minisgl/scheduler/scheduler.py) → [python/minisgl/scheduler/table.py](../../python/minisgl/scheduler/table.py)。追请求完成、延迟归还与表位置释放的关系。
- **负载与验收**：[lab/kv_cache/observe.py](../kv_cache/observe.py)、[tests/core/test_cache_allocate.py](../../tests/core/test_cache_allocate.py)。先读本题对应的负载和断言，明确观测对象。
- **暂时跳过**：attention kernel、host cache 和分布式同步。

### 做完后应看到

现有 pytest 正常完成的结尾形如：

```text
<实际数量> passed in <实际秒数>s
```

数量取决于测试参数化，以实际输出为准；这条命令不会打印前面 probe 的 `[check] PASS`。

**完成效果**：测试通过，另交一段真实 allocate→evict→allocate 的账本。格式：

| 时刻/操作 | 请求分配量 | 实际回收量 | available_size | evictable_size | protected_size | page_size |
|---|---:|---:|---:|---:|---:|---:|
| 操作前 | 填实测 | — | 填实测 | 填实测 | 填实测 | 填配置 |
| 淘汰后 | — | 填实测 | 填实测 | 填实测 | 填实测 | 同上 |
| 再分配后 | 填实测 | — | 填实测 | 填实测 | 填实测 | 同上 |

**失败/未完成**：pytest 报 FAILED 时保存失败场景；测试通过但把树中可淘汰量当成空闲 page 量、没有解释多回收量的去向，仍未完成本题。


In [ ]:
%%bash
export MINISGL_OBSERVE=0
python -m pytest -q -o addopts= tests/core/test_cache_allocate.py


## 【大改理论】Phase 4：新增主仓库的 LFU 淘汰能力

**大改理论题**：在 mini 只分析 LFU 的计数、节点拆分、锁与配置路径，并设计能区分 LRU/LFU 的负载；下面原有实现命令留作未来实作验收，不是转场门槛。


**外部行为要求**：默认策略仍按原有 LRU 行为工作；显式选择 `lfu` 时，优先淘汰复用次数较少的可淘汰分支，次数相同时保持稳定的先后规则。本实验把对已缓存前缀的再次 `insert_prefix` 视为一次复用。验收负载先插入 A、B，然后多次复用 A，最后一次复用 B：B 最近被访问，但 A 复用更多。默认策略淘汰 A，LFU 淘汰 B。已经锁住的分支在两种策略下都不能被淘汰；淘汰后剩余分支仍可匹配，缓存大小与 page 账本保持正确。

实验至少要支持 `RadixPrefixCache(..., eviction_policy="lfu")`，并能从离线 `LLM(..., radix_eviction_policy="lfu")` 选用它。只要求 `lru` 和 `lfu`；不新增完整的策略层次结构。

先答再改：哪些操作应被算作一次前缀复用？同一请求的 chunked prefill 会不会重复计数？节点拆分后，新旧分支的复用历史应如何解释？配置从 `LLM` 到缓存实例经过哪些已有层？主仓库如何将策略选择与缓存类型区分？

### 本题文件导航

- **策略与接口**：[python/minisgl/kvcache/radix_cache.py](../../python/minisgl/kvcache/radix_cache.py) → [python/minisgl/kvcache/base.py](../../python/minisgl/kvcache/base.py)。调查访问、拆分、锁定和淘汰在频率策略下需要遵守的语义。
- **配置到实例**：[python/minisgl/llm/llm.py](../../python/minisgl/llm/llm.py) → [python/minisgl/scheduler/config.py](../../python/minisgl/scheduler/config.py) → [python/minisgl/scheduler/scheduler.py](../../python/minisgl/scheduler/scheduler.py) → [python/minisgl/scheduler/cache.py](../../python/minisgl/scheduler/cache.py)。画出选项从离线入口到实际 prefix cache 实例的构造路线。
- **负载与验收**：[lab/kv_cache/observe.py](../kv_cache/observe.py)、[tests/core/test_cache_allocate.py](../../tests/core/test_cache_allocate.py)。先读本题对应的负载和断言，明确观测对象。
- **暂时跳过**：完整策略层次结构、HiCache 和 GPU attention。

### 做完后应看到

同一负载的关键输出应为（每条 probe 后各有一个 PASS）：

```text
[frequency] policy= default locked_b= False indices= [100, 101]
[check] PASS
[frequency] policy= lfu locked_b= False indices= [200, 201]
[check] PASS
[frequency] policy= lfu locked_b= True indices= [100, 101]
[frequency] size= SizeInfo(evictable_size=0, protected_size=2)
[check] PASS
[manager] recycled indices= [2, 3]
[check] PASS
```

前两组省略了 size 行，不能把省略理解成无须维护账本。默认与 LFU 都剩 2 个可淘汰 token；锁住 B 的组在解锁前有 2 个 protected token。

**通过标准**：策略、锁定、manager 回收和 split-lock 回归均通过；最后的离线 LLM 配置单元能生成一条 2-token 输出（原单元成功时不打印 PASS，查看断言与执行是否完成）。

**失败时观察**：不接受 `eviction_policy` / `radix_eviction_policy` 的启动异常是配置路径尚未接通的可观察症状；LFU 与默认淘汰相同，或锁住的 B 被回收，也不满足要求。


### Phase 4 的行为验收

前三个命令分别检查默认 LRU、LFU 和“锁定优先于淘汰策略”。第四个命令在 `CacheManager` 的 page 不足时触发淘汰，并核对回收 page 与账本；第五个命令检查此前的节点拆分与锁定行为没有回归。新增功能未实现时，带 `lfu` 的命令会失败；完成后应通过。不要只检查配置字段是否存在。


In [ ]:
%%bash
export MINISGL_OBSERVE=0
python lab/kv_cache/observe.py --case frequency
python lab/kv_cache/observe.py --case frequency --policy lfu
python lab/kv_cache/observe.py --case locked-frequency --policy lfu
python lab/kv_cache/observe.py --case manager
python lab/kv_cache/observe.py --case split-lock


### 配置路径检查（Linux CUDA 服务器）

完成 CPU 行为验收后，用已缓存的模型 config/tokenizer 跑一个离线请求，确认新选项可以从 `LLM` 进入实际缓存构造路径。这个 smoke test 只验证配置接入；策略是否真的改变淘汰对象，以上面的 CPU 行为验收为准。

### 本题文件导航

- **调用路线**：[python/minisgl/llm/llm.py](../../python/minisgl/llm/llm.py) → [python/minisgl/scheduler/config.py](../../python/minisgl/scheduler/config.py) → [python/minisgl/scheduler/scheduler.py](../../python/minisgl/scheduler/scheduler.py) → [python/minisgl/scheduler/cache.py](../../python/minisgl/scheduler/cache.py)。按构造顺序确认实际实例收到配置；不要把 smoke 成功当成策略行为证明。
- **负载与验收**：[lab/kv_cache/observe.py](../kv_cache/observe.py)。先读本题对应的负载和断言，明确观测对象。
- **暂时跳过**：权重映射和 CUDA Graph 内部。


In [ ]:
import os
os.environ["MINISGL_OBSERVE"] = "0"

import os
import platform
import torch
from minisgl.core import SamplingParams
from minisgl.llm import LLM

assert platform.system() == "Linux" and torch.cuda.is_available()
model = os.environ.get("MODEL_PATH", "Qwen/Qwen3-0.6B")
llm = LLM(model, use_dummy_weight=True, max_seq_len_override=512, radix_eviction_policy="lfu")
result = llm.generate([[100, 101, 102, 103]], SamplingParams(ignore_eos=True, max_tokens=2))
assert len(result) == 1 and len(result[0]["token_ids"]) == 2


## 对照 SGLang 主仓库

按这个顺序看：[公开选项](https://github.com/sgl-project/sglang/blob/main/python/sglang/srt/arg_groups/choices.py) → [淘汰策略](https://github.com/sgl-project/sglang/blob/main/python/sglang/srt/mem_cache/evict_policy.py) → [RadixCache](https://github.com/sgl-project/sglang/blob/main/python/sglang/srt/mem_cache/radix_cache.py) → [缓存初始化参数](https://github.com/sgl-project/sglang/blob/main/python/sglang/srt/mem_cache/cache_init_params.py)。只追 LRU/LFU、前缀复用、节点生命周期和策略配置。主仓库额外支持的策略、host cache、KV 事件、分布式通信先跳过。

复盘：mini 只支持两种策略时，哪些主仓库结构没有必要照搬？若以后要加更复杂的策略，你当前设计在哪些地方会开始不够用？


## 【大改理论】留作问题，不扩大本次实现

mini 的 `RadixPrefixCache.reset` 仍值得研究：

- 如果只清空 radix 树，已占用的 KV page 会由谁归还？在有活动请求持有 handle 时，清空树会发生什么？
- cache salt、TTL 或 session protection 会改变哪一层的契约？它们是否应该与 H 任务的 host 容量管理分开验收？

这一节只讨论 reset、salt、TTL 和 session protection 的边界；HiCache 与 MLA 的实现和验收已在本 lab 开篇单列为主任务。

### 本题文件导航

- **树与所有者**：[python/minisgl/kvcache/base.py](../../python/minisgl/kvcache/base.py) → [python/minisgl/kvcache/radix_cache.py](../../python/minisgl/kvcache/radix_cache.py) → [python/minisgl/scheduler/cache.py](../../python/minisgl/scheduler/cache.py)。追 reset、handle 生命周期与物理 page 所有权；此处只交理论分析。
- **负载与验收**：[lab/kv_cache/observe.py](../kv_cache/observe.py)。先读本题对应的负载和断言，明确观测对象。
- **暂时跳过**：在 reset 讨论中重做 H 任务的 host/offload 实现。

### 做完后应看到

这里不要求运行一个尚未实现的 reset 功能。把 Phase 0 的锁定基线当作证据，再交如下记录：

```text
讨论场景：<是否仍有活动请求持有 handle>
操作前：<树中的 indices / CacheManager 空闲量 / protected 量>
假设 reset 后：<树状态与物理 page 所有者分别会怎样>
最小反例：<操作顺序 + 预期暴露的错误>
依据文件：<实际读过的文件与接口契约>
```

**完成标准**：说明树被清空与物理 page 被归还的责任，并构造活动 handle 的边界反例；代码基线 PASS 不能代替这份分析。


## 【必做】源码原有 TODO/NOTE：缓存边界与树的不变量

以下标记来自 mini-sglang 本身。H0/M0 两项源码 TODO 基础已在开篇列为必做；H1/M1/M2 是大改理论。这里集中验证 NOTE 的边界；原有 Radix/LFU 练习仍可帮助解释 H/M 的资源行为。

| 源码标记 | 阅读与练习 |
|---|---|
| [`RadixPrefixCache.insert_prefix` 的 `NOTE: prefix_len < insert_len`](../../python/minisgl/kvcache/radix_cache.py) | 只在插入尚未覆盖的后缀时新建节点。用 Phase 0 的重复插入与分叉插入各画一次拓扑，核对新节点数量与 page indices。 |
| [`RadixPrefixCache.evict` 的 `NOTE: root is always protected`](../../python/minisgl/kvcache/radix_cache.py) | 在 Phase 2 的淘汰轨迹中标出根与候选叶；验证请求的淘汰量超过可淘汰量时会拒绝请求，而根与锁定分支始终不进入可回收 page 集合。 |
| [`RadixPrefixCache._tree_walk` 的 `NOTE: match_len >= page_size`](../../python/minisgl/kvcache/radix_cache.py) | 构造前缀只差不到一页与恰好一页的两组 token，比较 `match_prefix` 长度；解释按页对齐与压缩边匹配的关系。 |
| [`BasePrefixCache.evict` 文档中的两个 `Note`](../../python/minisgl/kvcache/base.py) | 用 `evict(0)` 和“请求量落在节点内部”两种情况检验：前者无动作，后者实际回收量可能大于请求量。把实际值带回 Phase 3 的 page 账本。 |
| [`MatchResult` 的 `TODO: support HiCache`](../../python/minisgl/kvcache/base.py) | 见开篇 H0/H1：必做当前 GPU 命中链与 host 缺口；host 恢复只交不变量和请求级测试计划。 |
| [`create_kv_cache_pool` 的 `TODO: ... MLA`](../../python/minisgl/kvcache/__init__.py) | 见开篇 M0–M2：必做 MHA 假设表；MLA pool 与真实模型对拍只交理论方案。 |
| [`tests/kernel/test_store.py` 的重复索引 `NOTE`](../../tests/kernel/test_store.py) | 这是 kernel 测试的输入约束。阅读该测试并说明重复索引会使被测写入行为的期望值为何不明确；它不是 radix 树去重的保证。 |

**记录**：至少保留一份“源码 NOTE → 输入 → 可观察结果 → 违反时会出现什么”的小表。TODO 基础以 H0/M0 的现状证据为转场标准；大改题保留将来的实现验收，NOTE 用边界输入验证。


## 交付自查

必做交付 H0/M0 的源码缺口与反例、Phase 0 的树形图、Phase 3 的 page 账本；H1/M1/M2 交理论方案。Phase 1/2 的改动和 Phase 4 的原有验收命令按标签处理，不作为转场门槛。必做运行 Phase 0 的 CPU probe 与 `tests/core/test_cache_allocate.py`；H/M 的请求级实现验收和原有配置 smoke test 留作后续选中对应任务时使用。记录一次缓存命中、一次节点拆分、一次锁保护、一次淘汰及其 page 账本变化。
